# LM Studio Llama 3.3 70B — `uni_subject` Annotation

**Model:** Llama 3.3 70B Instruct (Q4) via LM Studio

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os, time, re
from tqdm.auto import tqdm
from openai import OpenAI

LMSTUDIO_URL = "http://localhost:1234/v1"
MODEL_ID     = "meta/llama-3.3-70b"
TEMPERATURE  = 0
MAX_TOKENS   = 64
SEED         = 42

## 2. Set up client

In [ ]:
client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")
print("Models available in LM Studio:")
try:
    for m in client.models.list():
        print(f"  {m.id}")
except Exception as e:
    print(f"  Could not connect: {e}")

## 3. Load valid codes

In [ ]:
from corex_eval import load_training_data

train_df = load_training_data(
    task="annotation",
    variable="uni_subject",
    features=["subject_label", "uni_subject"],
)
train_df = train_df.dropna(subset=["subject_label", "uni_subject"]).reset_index(drop=True)

valid_codes = sorted(train_df["uni_subject"].unique().tolist())
codes_str   = "\n".join(f"  {c}" for c in valid_codes)
print(f"{len(valid_codes)} valid subject codes")

## 4. Build prompt & parser

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert political scientist coding university education data "
    "using the CoREx codebook.\n\n"
    "Task: given a subject description (combining degree type and field of study), "
    "assign the single most appropriate subject code.\n"
    "Respond with ONLY the exact code label as listed below — nothing else.\n\n"
    f"Valid subject codes:\n{codes_str}"
)

def parse_prediction(text, valid_codes):
    import re
    text = text.strip().strip("\"'")
    # Exact match
    if text in valid_codes:
        return text
    # Build map: leading digits → full code (e.g. "03" → "03 – Social sciences")
    code_map = {}
    for c in valid_codes:
        m = re.match(r'^(\d+)', c)
        if m:
            code_map[m.group(1)] = c
    # Extract leading digits from prediction
    m = re.match(r'^(\d+)', text)
    if m and m.group(1) in code_map:
        return code_map[m.group(1)]
    # Fallback: substring match on description
    text_lower = text.lower()
    for code in valid_codes:
        parts = re.split(r'\s*[\u2013=]\s*', code, maxsplit=1)
        desc = parts[1].lower() if len(parts) > 1 else code.lower()
        if desc in text_lower or text_lower in desc:
            return code
    return text

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")


## 5. Load test inputs

In [ ]:
from corex_eval import load_inputs

test_df = load_inputs(task="annotation", variable="uni_subject")
print(f"{len(test_df)} test rows")
test_df.head()

## 6. Run inference

In [ ]:
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    subject  = str(row["subject_label"] or "")
    user_msg = f"Subject description: {subject}"
    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=TEMPERATURE, max_tokens=MAX_TOKENS, seed=SEED,
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

## 7. Evaluate

In [ ]:
import pandas as pd
from corex_eval import evaluate

results = evaluate(
    predictions_df,
    task="annotation",
    variable="uni_subject",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/education/lmstudio_llama33_70b_uni_subject/config.yaml",
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})